<a href="https://colab.research.google.com/github/lohaniSatwik/steam-games-data-mining/blob/master/Code/section4c_xgboost_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section 4c — XGBoost
**IE500 Data Mining | Team 9 – Brewed Clusters**

> **Google Colab notebook.** Run all cells top to bottom.

### Setup
- Model: `XGBClassifier` — gradient boosting on decision trees
- No scaling needed — tree-based model
- **Imbalance:** `sample_weight` computed via `compute_sample_weight('balanced')` — XGBoost has no `class_weight` parameter, so we pass per-sample weights instead
- **Outer CV:** 5-fold stratified (unbiased performance estimate)
- **Inner CV:** 3-fold `GridSearchCV` over `n_estimators`, `max_depth`, `learning_rate`
- **Metric:** Macro F1 (primary), per-class F1 (secondary)
- **Baselines to beat:** LR = 0.4355 | SVM = 0.4693

### Why XGBoost?
XGBoost builds trees **sequentially** — each new tree corrects the errors of the previous ones.  
This is different from Random Forest (parallel trees, bagging). XGBoost typically achieves  
higher accuracy on tabular data but is more sensitive to hyperparameters.

In [ ]:
!pip install xgboost -q

In [ ]:
import os
if not os.path.exists('steam-games-data-mining'):
    !git clone https://github.com/lohaniSatwik/steam-games-data-mining.git
else:
    !git -C steam-games-data-mining pull
DATA_DIR = 'steam-games-data-mining/Datasets'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from collections import Counter
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE  = 42
CLASS_ORDER   = ['Good', 'Mixed', 'Bad']
CLASS_COLORS  = {'Good': 'steelblue', 'Mixed': 'sandybrown', 'Bad': 'salmon'}

BASELINE_LR  = 0.4355
BASELINE_SVM = 0.4693

print('Libraries loaded.')

In [ ]:
train = pd.read_csv(f'{DATA_DIR}/train_multiclass.csv')
test  = pd.read_csv(f'{DATA_DIR}/test_multiclass.csv')

X_train = train.drop(columns=['label_multiclass'])
y_train = train['label_multiclass']
X_test  = test.drop(columns=['label_multiclass'])
y_test  = test['label_multiclass']

# XGBoost requires numeric labels — encode Good/Mixed/Bad → 0/1/2
le = LabelEncoder()
le.fit(CLASS_ORDER)
y_train_enc = le.transform(y_train)
y_test_enc  = le.transform(y_test)

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}')
print('\nClass distribution (train):')
vc = y_train.value_counts()
for cls in CLASS_ORDER:
    print(f'  {cls:6s}: {vc[cls]:6,d}  ({vc[cls]/len(y_train)*100:.1f}%)')

## Nested Cross-Validation

- **Outer loop** (5 folds) — unbiased estimate of generalisation performance
- **Inner loop** (3-fold `GridSearchCV`) — selects best hyperparameters
- **`n_estimators`** — number of boosting rounds (trees)
- **`max_depth`** — depth of each tree; shallow trees = simpler model
- **`learning_rate`** — how much each tree corrects the previous ones; lower = more trees needed but better generalisation
- `sample_weight` is recomputed per outer fold and passed to `gs.fit()` — sklearn subsets it correctly for inner folds

Expected runtime on Colab: **~15–30 minutes**

In [ ]:
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

param_grid = {
    'n_estimators':  [100, 200],
    'max_depth':     [3, 6],
    'learning_rate': [0.05, 0.1]
}

outer_scores    = []
best_params_log = []

print('Running 5-fold nested CV (inner 3-fold GridSearchCV)...\n')

for fold, (tr_idx, val_idx) in tqdm(
        enumerate(outer_cv.split(X_train, y_train_enc), 1),
        total=5, desc='Outer folds'):

    X_tr, X_val       = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val       = y_train_enc[tr_idx],  y_train_enc[val_idx]
    sample_weight_tr  = compute_sample_weight('balanced', y_tr)

    base_clf = XGBClassifier(
        objective='multi:softmax',
        num_class=3,
        eval_metric='mlogloss',
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0
    )
    gs = GridSearchCV(
        base_clf, param_grid,
        cv=inner_cv, scoring='f1_macro',
        n_jobs=-1, refit=True
    )
    gs.fit(X_tr, y_tr, sample_weight=sample_weight_tr)

    y_pred = gs.predict(X_val)
    f1 = f1_score(y_val, y_pred, average='macro')
    outer_scores.append(f1)
    best_params_log.append(gs.best_params_)

    print(f'  Fold {fold} | Macro F1: {f1:.4f} | {gs.best_params_}')

print(f'\nNested CV  →  Macro F1: {np.mean(outer_scores):.4f} ± {np.std(outer_scores):.4f}')
print(f'Baseline LR           →  Macro F1: {BASELINE_LR:.4f}')
print(f'Baseline SVM          →  Macro F1: {BASELINE_SVM:.4f}')
print(f'Improvement over SVM  →  {np.mean(outer_scores) - BASELINE_SVM:+.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
folds = list(range(1, 6))
ax.bar(folds, outer_scores, color='forestgreen', edgecolor='white', alpha=0.85)
ax.axhline(np.mean(outer_scores), color='crimson', linestyle='--', linewidth=1.5,
           label=f'XGB Mean = {np.mean(outer_scores):.4f}')
ax.axhline(BASELINE_SVM, color='darkorange', linestyle='--', linewidth=1.5,
           label=f'SVM Baseline = {BASELINE_SVM:.4f}')
ax.axhline(BASELINE_LR, color='grey', linestyle='--', linewidth=1.5,
           label=f'LR Baseline = {BASELINE_LR:.4f}')
ax.axhline(np.mean(outer_scores) + np.std(outer_scores), color='crimson',
           linestyle=':', linewidth=1, alpha=0.6)
ax.axhline(np.mean(outer_scores) - np.std(outer_scores), color='crimson',
           linestyle=':', linewidth=1, alpha=0.6,
           label=f'±1 std = {np.std(outer_scores):.4f}')
ax.set_xlabel('Outer Fold')
ax.set_ylabel('Macro F1')
ax.set_title('XGBoost — Nested CV Macro F1 per Fold')
ax.set_xticks(folds)
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
print('Best hyperparameters across outer folds:')
param_counts = Counter([str(p) for p in best_params_log])
for params, count in param_counts.most_common():
    print(f'  {count:2d} fold(s): {params}')

## Final Model

Re-run `GridSearchCV` on the **full training set**, then evaluate on the held-out test set **once**.

In [ ]:
print('Fitting final model on full training set...\n')

sample_weight_full = compute_sample_weight('balanced', y_train_enc)

final_gs = GridSearchCV(
    XGBClassifier(
        objective='multi:softmax',
        num_class=3,
        eval_metric='mlogloss',
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0
    ),
    param_grid,
    cv=inner_cv, scoring='f1_macro',
    n_jobs=-1, refit=True
)
final_gs.fit(X_train, y_train_enc, sample_weight=sample_weight_full)

print(f'Best params      : {final_gs.best_params_}')
print(f'Best inner CV F1 : {final_gs.best_score_:.4f}')

## Test Set Evaluation

> Evaluate on `test_multiclass.csv` **once only** — this is the final performance number.

In [ ]:
final_model  = final_gs.best_estimator_
y_pred_enc   = final_model.predict(X_test)
y_pred_test  = le.inverse_transform(y_pred_enc.astype(int))

test_macro_f1 = f1_score(y_test, y_pred_test, average='macro')
print(f'Test set Macro F1 : {test_macro_f1:.4f}')
print(f'LR Baseline       : {BASELINE_LR:.4f}')
print(f'SVM Baseline      : {BASELINE_SVM:.4f}')
print(f'Improvement vs SVM: {test_macro_f1 - BASELINE_SVM:+.4f}\n')
print('Classification Report (Test Set):')
print(classification_report(y_test, y_pred_test, labels=CLASS_ORDER, target_names=CLASS_ORDER))

In [ ]:
cm = confusion_matrix(y_test, y_pred_test, labels=CLASS_ORDER)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_ORDER)
disp.plot(ax=ax, colorbar=False, cmap='Greens')
ax.set_title('XGBoost — Confusion Matrix (Test Set)', pad=12)
plt.tight_layout()
plt.show()

## Feature Importance

XGBoost provides feature importance by **gain** — the average improvement in loss brought by a feature across all splits where it is used.  
Gain is more reliable than frequency-based importance for identifying truly predictive features.

In [ ]:
feature_names = X_train.columns.tolist()
imp_series = pd.Series(
    final_model.feature_importances_,
    index=feature_names
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
imp_series.head(20).sort_values().plot(
    kind='barh', ax=ax, color='forestgreen', alpha=0.85
)
ax.set_title('XGBoost — Top 20 Feature Importances (Gain)', fontsize=13)
ax.set_xlabel('Feature Importance (Gain)')
plt.tight_layout()
plt.show()

print('\nTop 10 features:')
for feat, imp in imp_series.head(10).items():
    print(f'  {feat:40s}  {imp:.4f}')

## Results Summary

| Metric | Value |
|--------|-------|
| Nested CV Macro F1 (mean ± std) | — |
| Test set Macro F1 | — |
| Best params (nested CV) | — |
| Best params (final model) | — |
| vs LR Baseline (0.4355) | — |
| vs SVM Baseline (0.4693) | — |

**Per-class F1 (test set):**

| Class | Precision | Recall | F1 |
|-------|-----------|--------|----|
| Good  | — | — | — |
| Mixed | — | — | — |
| Bad   | — | — | — |

**Interpretation:**

> Fill in after running.